# Contrôle de reproductibilité du paquet Vatancul

Ce notebook conserve uniquement les opérations utiles pour **contrôler techniquement une livraison** :

1. inventaire des fichiers ;
2. vérification des empreintes SHA-256 ;
3. contrôle des rasters ;
4. contrôle des GeoPackages ;
5. contrôle simple du bassin et de l'exutoire ;
6. inventaire des ressources de publication WebSIG.

Les cellules de travail interne qui inspectaient ou réécrivaient automatiquement le mémoire Word ont été supprimées.  
Les anciennes phrases de statut sur HEC-RAS, MapStore ou des analyses non finalisées ne sont pas reprises : ce notebook sert uniquement au **contrôle des fichiers et de leur cohérence technique**.


## 1. Configuration

In [ ]:
from pathlib import Path
import hashlib
import json

import pandas as pd
import geopandas as gpd
import rasterio
import fiona

# ------------------------------------------------------------------
# À ADAPTER
# ------------------------------------------------------------------
BASE = Path(r"D:\CHEMIN\VERS\vatencul_lidar_websig_outputs")

MANIFEST_SHA256 = BASE / "manifest_sha256.csv"
GPKG = BASE / "vectors" / "vatencul_objets_websig.gpkg"

RASTER_DIR = BASE / "rasters"
WEB_DIR = BASE / "mapstore_geoserver"
COG_DIR = WEB_DIR / "cog"

REPORT_DIR = BASE / "controle_reproductibilite"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

EPSG_ATTENDU = 2154


## 2. Inventaire de la livraison

In [ ]:
rows = []

for path in sorted(BASE.rglob("*")):
    if path.is_file():
        rows.append({
            "fichier": str(path.relative_to(BASE)),
            "extension": path.suffix.lower(),
            "taille_octets": path.stat().st_size,
            "taille_mio": path.stat().st_size / 1024**2,
        })

inventory = pd.DataFrame(rows)

print("Nombre de fichiers :", len(inventory))
print("Taille totale (Mio) :", round(inventory["taille_mio"].sum(), 2))

inventory.to_csv(
    REPORT_DIR / "inventaire_fichiers.csv",
    index=False,
)

inventory.head(20)


## 3. Vérification des empreintes SHA-256

Le manifeste est utilisé uniquement pour contrôler l'intégrité des fichiers présents.  
Une empreinte valide signifie que le fichier correspond au contenu enregistré dans le manifeste ; elle ne constitue pas une validation scientifique du résultat.


In [ ]:
def sha256_file(path: Path, chunk_size=1024 * 1024) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)

    return h.hexdigest()


if not MANIFEST_SHA256.exists():
    print("Manifest SHA-256 absent :", MANIFEST_SHA256)

else:
    manifest = pd.read_csv(MANIFEST_SHA256)

    path_col = next(
        c for c in manifest.columns
        if "fichier" in c.lower() or "path" in c.lower()
    )
    hash_col = next(
        c for c in manifest.columns
        if "sha" in c.lower()
    )

    checks = []

    for _, row in manifest.iterrows():
        rel = str(row[path_col])
        expected = str(row[hash_col]).strip().lower()
        path = BASE / rel

        exists = path.exists() and path.is_file()
        actual = sha256_file(path) if exists else None

        checks.append({
            "fichier": rel,
            "present": exists,
            "sha256_attendu": expected,
            "sha256_calcule": actual,
            "concordant": exists and actual == expected,
        })

    sha_report = pd.DataFrame(checks)

    print(
        "Empreintes concordantes :",
        int(sha_report["concordant"].sum()),
        "/",
        len(sha_report),
    )

    if not sha_report["concordant"].all():
        print("\nFichiers absents ou différents :")
        display(
            sha_report.loc[
                ~sha_report["concordant"],
                ["fichier", "present", "concordant"],
            ]
        )

    sha_report.to_csv(
        REPORT_DIR / "controle_sha256.csv",
        index=False,
    )


## 4. Contrôle des rasters

In [ ]:
raster_rows = []

raster_paths = []
if RASTER_DIR.exists():
    raster_paths.extend(RASTER_DIR.glob("*.tif"))
if COG_DIR.exists():
    raster_paths.extend(COG_DIR.glob("*.tif"))

for path in sorted(set(raster_paths)):
    with rasterio.open(path) as ds:
        raster_rows.append({
            "fichier": str(path.relative_to(BASE)),
            "driver": ds.driver,
            "crs": ds.crs.to_string() if ds.crs else None,
            "resolution_x": abs(ds.transform.a),
            "resolution_y": abs(ds.transform.e),
            "largeur": ds.width,
            "hauteur": ds.height,
            "nombre_bandes": ds.count,
            "dtype": ds.dtypes[0],
            "nodata": ds.nodata,
            "tuilé": ds.is_tiled,
            "overviews": ",".join(map(str, ds.overviews(1))),
            "xmin": ds.bounds.left,
            "ymin": ds.bounds.bottom,
            "xmax": ds.bounds.right,
            "ymax": ds.bounds.top,
        })

raster_report = pd.DataFrame(raster_rows)

if len(raster_report):
    print("Rasters contrôlés :", len(raster_report))

    if "crs" in raster_report:
        print("\nCRS rencontrés :")
        print(raster_report["crs"].value_counts(dropna=False))

    display(
        raster_report[
            [
                "fichier",
                "driver",
                "crs",
                "resolution_x",
                "resolution_y",
                "largeur",
                "hauteur",
                "nodata",
            ]
        ]
    )

    raster_report.to_csv(
        REPORT_DIR / "controle_rasters.csv",
        index=False,
    )

else:
    print("Aucun raster trouvé.")


## 5. Contrôle du GeoPackage

In [ ]:
gpkg_rows = []

if not GPKG.exists():
    print("GeoPackage absent :", GPKG)

else:
    layers = fiona.listlayers(GPKG)
    print("Nombre de couches :", len(layers))

    for layer in layers:
        gdf = gpd.read_file(GPKG, layer=layer)

        gpkg_rows.append({
            "couche": layer,
            "nb_entites": len(gdf),
            "crs": gdf.crs.to_string() if gdf.crs else None,
            "geometries_valides": (
                int(gdf.geometry.is_valid.sum())
                if len(gdf) else 0
            ),
            "geometries_vides": (
                int(gdf.geometry.is_empty.sum())
                if len(gdf) else 0
            ),
            "types_geometriques": (
                ", ".join(
                    f"{k}:{v}"
                    for k, v in
                    gdf.geom_type.value_counts().to_dict().items()
                )
                if len(gdf) else ""
            ),
        })

    gpkg_report = pd.DataFrame(gpkg_rows)
    display(gpkg_report)

    gpkg_report.to_csv(
        REPORT_DIR / "controle_geopackage.csv",
        index=False,
    )


## 6. Contrôle simple du bassin et de l'exutoire

Cette étape vérifie uniquement la géométrie et la relation spatiale entre le bassin et l'exutoire.  
Elle ne constitue pas une validation hydrologique ou hydraulique.


In [ ]:
if GPKG.exists():
    layers = set(fiona.listlayers(GPKG))

    if {"bassin", "exutoire"}.issubset(layers):
        bassin = gpd.read_file(GPKG, layer="bassin")
        exutoire = gpd.read_file(GPKG, layer="exutoire")

        if bassin.crs and bassin.crs.to_epsg() != EPSG_ATTENDU:
            bassin = bassin.to_crs(EPSG_ATTENDU)

        if exutoire.crs and exutoire.crs.to_epsg() != EPSG_ATTENDU:
            exutoire = exutoire.to_crs(EPSG_ATTENDU)

        bassin_union = bassin.geometry.union_all()

        print(
            "Surface du bassin :",
            round(bassin_union.area / 1e4, 3),
            "ha",
        )
        print(
            "Bassin valide :",
            bool(bassin.geometry.is_valid.all()),
        )

        if len(exutoire):
            exut_geom = exutoire.geometry.iloc[0]

            print(
                "Exutoire couvert par le bassin :",
                bool(bassin_union.covers(exut_geom)),
            )
            print(
                "Distance exutoire-limite :",
                round(
                    bassin_union.boundary.distance(exut_geom),
                    3,
                ),
                "m",
            )
    else:
        print(
            "Les couches 'bassin' et/ou 'exutoire' "
            "ne sont pas présentes."
        )


## 7. Ressources de publication WebSIG

In [ ]:
web_candidates = [
    WEB_DIR / "layer_catalog.json",
    WEB_DIR / "mapstore_config_template.json",
    WEB_DIR / "batiments_extrusion.geojson",
]

web_rows = []

for path in web_candidates:
    web_rows.append({
        "fichier": str(path.relative_to(BASE)),
        "present": path.exists(),
        "taille_mio": (
            path.stat().st_size / 1024**2
            if path.exists() else None
        ),
    })

web_report = pd.DataFrame(web_rows)
display(web_report)

web_report.to_csv(
    REPORT_DIR / "controle_ressources_websig.csv",
    index=False,
)

catalog_path = WEB_DIR / "layer_catalog.json"

if catalog_path.exists():
    catalog = json.loads(
        catalog_path.read_text(encoding="utf-8")
    )

    print(
        "\nEntrées du catalogue :",
        len(catalog) if isinstance(catalog, list)
        else "structure non tabulaire",
    )

    if isinstance(catalog, list):
        for item in catalog:
            print(
                "-",
                item.get("id"),
                "|",
                item.get("title"),
                "|",
                item.get("service"),
            )


## 8. Synthèse du contrôle

In [ ]:
summary = {
    "nombre_fichiers": len(inventory),
    "taille_totale_mio": round(
        inventory["taille_mio"].sum(),
        3,
    ),
    "manifest_sha256_present": MANIFEST_SHA256.exists(),
    "geopackage_present": GPKG.exists(),
    "nombre_rasters_controles": (
        len(raster_report)
        if "raster_report" in globals()
        else 0
    ),
    "nombre_couches_gpkg": (
        len(gpkg_report)
        if "gpkg_report" in globals()
        else 0
    ),
}

if "sha_report" in globals():
    summary["sha256_total"] = len(sha_report)
    summary["sha256_concordants"] = int(
        sha_report["concordant"].sum()
    )

summary_df = pd.DataFrame([summary])
display(summary_df)

summary_df.to_csv(
    REPORT_DIR / "synthese_controle.csv",
    index=False,
)


## Interprétation

Ce notebook répond à une question simple : **les fichiers livrés sont-ils présents, lisibles et techniquement cohérents avec leur organisation annoncée ?**

Il ne doit pas être utilisé pour conclure sur :

- la validité hydraulique d'une simulation ;
- l'exactitude absolue du MNT ;
- la performance réelle d'un serveur GeoServer ou de MapStore ;
- l'efficacité de la 3D pour les utilisateurs ;
- la validité scientifique d'un résultat uniquement parce que son empreinte SHA-256 est correcte.

Ces évaluations nécessitent leurs propres protocoles.
